In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op='', label='a'):
        self.dat = data
        self.grad = 0.0
        self._op = _op
        self._prev = set(_prev)
        self._backward = lambda: None
        self.label = label

    def __add__(self, other):
        out = Value(self.data + other.data, _op='+', _prev=(self, other))

        def _backward():
            self.grad = 1*out.grad
            other.grad = 1*out.grad

        out._backward = _backward
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, _op='*', _prev=(self, other))

        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad

        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    
    def __str__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __repr__(self):
        return self.__str__()

In [4]:
a = Value(4, label='a')
b = Value(-5, label='b')
c = a * b; c.label = 'c'
d =Value(3, label='d')
e = d + c; e.label = 'e'
f = Value(-6, label='f')
o = e * f; o.label = 'o'

AttributeError: 'Value' object has no attribute 'data'

In [7]:
import torch
import numpy as np

In [12]:
X = torch.tensor([[1, 2], [3, 4]], dtype=torch.float)
W = torch.tensor([[11, 21], [31, 41]], dtype=torch.float, requires_grad=True)


In [13]:
c = X @ W

In [14]:
o = torch.sum(c)

In [15]:
o

tensor(560., grad_fn=<SumBackward0>)

In [16]:
W.grad

In [17]:
o.backward()

In [19]:
W.grad

tensor([[4., 4.],
        [6., 6.]])

In [1]:
class Value:
    def __init__(self, data, _prev=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._op = _op
        self._prev = set(_prev)
        self._backward = lambda: None
        self.label = label

    def __add__(self, other):
        out = Value(self.data + other.data, _prev=(self, other), _op='+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, _prev=(self, other), _op='*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad}, label={self.label})"


In [2]:
a = Value(4, label='a')
b = Value(-5, label='b')
c = a * b; c.label = 'c'

d = Value(3, label='d')
e = d + c; e.label = 'e'

f = Value(-6, label='f')
o = e * f; o.label = 'o'


In [3]:
from graphviz import Digraph

def trace(root):
    nodes = set()
    edges = set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_graph(root):
    dot = Digraph(format='png', graph_attr={'rankdir': 'LR'})

    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))
        label = f"{n.label} | data={n.data} | grad={n.grad}"
        dot.node(name=uid, label=label, shape='record')

        if n._op:
            op_id = uid + n._op
            dot.node(op_id, label=n._op)
            dot.edge(op_id, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2))+n2._op)

    return dot


In [5]:
o.backward()
dot = draw_graph(o)
dot.render("graph", view=True)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH